# SOLAR: multivariate conditional kernel-density models

This notebook implements the statistical core of **SOLAR (Statistical Optimisation for Localised Anomaly Recognition)**. It estimates the conditional distribution of a target pressure sensor from one or more correlated sensors, then identifies the highest-density region containing a chosen probability mass.

The confidence region can contain several disjoint intervals when the conditional distribution is multimodal. An observation between two high-density modes is therefore anomalous even if it lies inside the overall minimum/maximum envelope.

> The full 19,201-query by 10,000-grid calculation described in the report is computationally expensive. The default run evaluates one representative multivariate query; the last section retains an explicit, opt-in batch path.

## 1. Setup and data loading

In [ ]:
from itertools import combinations
from pathlib import Path
import json
import os

MPL_CONFIG_DIR = Path.cwd() / "tmp" / "matplotlib"
MPL_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CONFIG_DIR))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.integrate import trapezoid
from scipy.spatial.distance import cdist
from scipy.stats import norm

ROOT = Path.cwd()
DATA_PATH = Path(os.environ.get("SOLAR_DATA_PATH", ROOT / "march_26_data_corrected_table.xlsx"))
CORRELATION_PATH = ROOT / "highly_correlated_sensors.csv"
MODEL_OUTPUT_DIR = ROOT / "model_outputs"
ASSET_DIR = ROOT / "assets"
ASSET_DIR.mkdir(exist_ok=True)

CONFIDENCE = 0.99
TARGET_SENSOR = "P01"
INPUT_SENSORS = ("P10", "P11")
QUERY_INDEX = 10_000
GRID_SIZE = 2_000  # Use 10_000 to match the full-resolution experiments.
plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Sensor data not found at {DATA_PATH}. Set SOLAR_DATA_PATH to an authorised local copy."
    )

sensor_df = pd.read_excel(DATA_PATH).drop(columns=["Timetick"], errors="ignore")
correlated_df = pd.read_csv(CORRELATION_PATH)
required_sensors = {TARGET_SENSOR, *INPUT_SENSORS}
missing_sensors = required_sensors.difference(sensor_df.columns)
if missing_sensors:
    raise KeyError(f"Missing sensor columns: {sorted(missing_sensors)}")
if sensor_df[list(required_sensors)].isna().any().any():
    raise ValueError("The selected model columns contain missing values.")

print(f"Loaded {len(sensor_df):,} observations across {sensor_df.shape[1]} sensor series.")
display(
    correlated_df[correlated_df["Sensor1"].eq(TARGET_SENSOR)]
    .sort_values("Correlation", ascending=False)
    .head(5)
)

## 2. Conditional KDE and highest-density confidence regions

For a query vector $x_q$, Gaussian input-kernel weights identify historically similar network states. A weighted output KDE then estimates $f(y\mid x_q)$. Grid points are ordered by density and accumulated until the requested probability mass is reached. Reapplying that selection to the original output grid produces one or more contiguous, potentially asymmetric confidence regions.

In [ ]:
def silverman_bandwidth(values):
    """Return the fixed bandwidth used in the original implementation."""
    values = np.asarray(values, dtype=float)
    sigma = np.std(values, ddof=1)
    iqr = np.subtract(*np.percentile(values, [75, 25]))
    scale = min(sigma, iqr / 1.34)
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError("Bandwidth cannot be estimated from this series.")
    return 0.9 * scale * len(values) ** (-1 / 5)


def conditional_kde_pdf(X_train, y_train, X_query, y_grid, reg_bandwidth, ker_bandwidth, grid_chunk_size=250):
    """Estimate f(y|x) for one multivariate query without constructing the full Q x N surface."""
    X_train = np.asarray(X_train, dtype=float)
    X_query = np.asarray(X_query, dtype=float).reshape(1, -1)
    y_train = np.asarray(y_train, dtype=float).reshape(-1)
    y_grid = np.asarray(y_grid, dtype=float).reshape(-1)
    if X_train.ndim != 2 or X_train.shape[0] != y_train.size:
        raise ValueError("X_train and y_train have incompatible shapes.")
    if X_query.shape[1] != X_train.shape[1]:
        raise ValueError("X_query must have the same number of features as X_train.")
    if reg_bandwidth <= 0 or ker_bandwidth <= 0:
        raise ValueError("Kernel bandwidths must be positive.")

    input_distances = cdist(X_query, X_train, metric="euclidean").ravel()
    input_weights = norm.pdf(input_distances / reg_bandwidth)
    if not np.isfinite(input_weights).all() or input_weights.sum() <= np.finfo(float).tiny:
        raise FloatingPointError("All input-kernel weights underflowed; increase reg_bandwidth.")

    density = np.empty_like(y_grid)
    for start in range(0, y_grid.size, grid_chunk_size):
        stop = min(start + grid_chunk_size, y_grid.size)
        output_weights = norm.pdf((y_grid[start:stop, None] - y_train[None, :]) / ker_bandwidth)
        density[start:stop] = output_weights @ input_weights
    area = trapezoid(density, y_grid)
    if not np.isfinite(area) or area <= 0:
        raise FloatingPointError("The conditional density could not be normalised.")
    return density / area


def highest_density_regions(y_grid, density, confidence=0.99):
    """Return the selected grid mask, disjoint intervals, and density cutoff."""
    if not 0 < confidence <= 1:
        raise ValueError("confidence must be in (0, 1].")
    y_grid = np.asarray(y_grid, dtype=float)
    density = np.asarray(density, dtype=float)
    if y_grid.ndim != 1 or density.shape != y_grid.shape or y_grid.size < 2:
        raise ValueError("y_grid and density must be matching one-dimensional arrays.")

    cell_mass = density / density.sum()  # Equally spaced output grid.
    density_order = np.argsort(density)[::-1]
    cumulative_mass = np.cumsum(cell_mass[density_order])
    final_rank = min(np.searchsorted(cumulative_mass, confidence, side="left"), y_grid.size - 1)
    accepted = np.zeros(y_grid.size, dtype=bool)
    accepted[density_order[: final_rank + 1]] = True
    transitions = np.diff(np.pad(accepted.astype(int), (1, 1)))
    starts = np.where(transitions == 1)[0]
    ends = np.where(transitions == -1)[0] - 1
    intervals = [(float(y_grid[start]), float(y_grid[end])) for start, end in zip(starts, ends)]
    return accepted, intervals, float(density[density_order[final_rank]])


def value_in_regions(value, intervals):
    return any(lower <= value <= upper for lower, upper in intervals)

## 3. Representative two-input model

The supplied update fixed its visual example at query index 10,000. This keeps that reference point but evaluates the genuinely multivariate P10/P11 → P01 model and safely handles shorter datasets. The shared Silverman bandwidth follows the supplied implementation.

In [ ]:
X_train = sensor_df[list(INPUT_SENSORS)].to_numpy()
y_train = sensor_df[TARGET_SENSOR].to_numpy()
query_index = min(QUERY_INDEX, len(sensor_df) - 1)
bandwidth = silverman_bandwidth(y_train)
output_span = np.ptp(y_train)
y_grid = np.linspace(y_train.min() - 0.2 * output_span, y_train.max() + 0.2 * output_span, GRID_SIZE)
density = conditional_kde_pdf(
    X_train, y_train, X_train[query_index], y_grid, reg_bandwidth=bandwidth, ker_bandwidth=bandwidth
)
accepted, confidence_regions, density_cutoff = highest_density_regions(y_grid, density, CONFIDENCE)
actual_value = float(y_train[query_index])
is_anomaly = not value_in_regions(actual_value, confidence_regions)

print(f"Query index: {query_index:,}")
print(f"Inputs {INPUT_SENSORS}: {X_train[query_index]}")
print(f"Observed {TARGET_SENSOR}: {actual_value:.6f}")
print(f"Bandwidth: {bandwidth:.8f}")
print(f"{CONFIDENCE:.0%} highest-density regions: {confidence_regions}")
print(f"Anomaly: {is_anomaly}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(y_grid, density, color="#255f85", linewidth=2, label="Conditional KDE")
axes[0].fill_between(y_grid, 0, density, where=accepted, color="#5ba66b", alpha=0.35, label=f"{CONFIDENCE:.0%} region")
for number, (lower, upper) in enumerate(confidence_regions):
    axes[0].axvline(lower, color="#cc5500", linestyle="--", linewidth=1.2, label="Region boundaries" if number == 0 else None)
    axes[0].axvline(upper, color="#cc5500", linestyle="--", linewidth=1.2)
axes[0].axvline(actual_value, color="#202020", linewidth=1.5, label="Observed P01")
axes[0].set(title="Conditional density and confidence regions", xlabel="P01 pressure", ylabel="Density")
axes[0].legend(frameon=True)

ordered_mass = (density / density.sum())[np.argsort(density)[::-1]]
axes[1].plot(np.cumsum(ordered_mass), color="#255f85", linewidth=2)
axes[1].axhline(CONFIDENCE, color="#cc5500", linestyle="--", label=f"{CONFIDENCE:.0%} threshold")
axes[1].set(title="Mass accumulated by density rank", xlabel="Grid points ordered by density", ylabel="Cumulative probability")
axes[1].set_ylim(0, 1.02)
axes[1].legend(frameon=True)
fig.suptitle(f"Multivariate model: {', '.join(INPUT_SENSORS)} → {TARGET_SENSOR}", fontweight="bold")
fig.tight_layout()
figure_path = ASSET_DIR / "conditional-confidence-regions.png"
fig.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved {figure_path.relative_to(ROOT)}")

## 4. Optional batch export

The original notebook constructed the complete dense query-by-output surface. The function below evaluates one query at a time, avoiding the largest allocation but not the underlying computational cost. `LowerBound` and `UpperBound` remain as a compatibility envelope; `ConfidenceRegions` records every disjoint interval and drives the anomaly flag.

In [ ]:
def train_models(sensor_list, input_count, *, query_indices, bandwidth=None, confidence=0.99, grid_size=2_000, output_dir=MODEL_OUTPUT_DIR):
    """Train selected n-input models and export query-level confidence regions."""
    query_indices = np.asarray(query_indices, dtype=int)
    if query_indices.size == 0:
        raise ValueError("Provide at least one query index.")
    if query_indices.min() < 0 or query_indices.max() >= len(sensor_df):
        raise IndexError("A query index is outside the available dataset.")

    for sensor in sensor_list:
        candidates = (
            correlated_df[correlated_df["Sensor1"].eq(sensor)]
            .sort_values("Correlation", ascending=False).head(5)["Sensor2"].tolist()
        )
        if len(candidates) < input_count:
            continue
        for combo in combinations(candidates, input_count):
            X_values = sensor_df[list(combo)].to_numpy()
            y_values = sensor_df[sensor].to_numpy()
            model_bandwidth = bandwidth or silverman_bandwidth(y_values)
            span = np.ptp(y_values)
            model_grid = np.linspace(y_values.min() - 0.2 * span, y_values.max() + 0.2 * span, grid_size)
            rows = []
            for index in query_indices:
                query_density = conditional_kde_pdf(
                    X_values, y_values, X_values[index], model_grid, model_bandwidth, model_bandwidth
                )
                _, regions, _ = highest_density_regions(model_grid, query_density, confidence)
                actual = float(y_values[index])
                rows.append({
                    "QueryIndex": int(index), "Actual": actual,
                    "LowerBound": min(lower for lower, _ in regions),
                    "UpperBound": max(upper for _, upper in regions),
                    "ConfidenceRegions": json.dumps(regions),
                    "Anomaly": int(not value_in_regions(actual, regions)),
                })
            destination = output_dir / f"{input_count}-input" / sensor
            destination.mkdir(parents=True, exist_ok=True)
            filename = f"{sensor}_vs_{'_'.join(combo)}.csv"
            pd.DataFrame(rows).to_csv(destination / filename, index=False)
            print(f"Saved {sensor} with inputs {combo}: {len(rows):,} queries")


RUN_BATCH_EXPORT = False
if RUN_BATCH_EXPORT:
    selected_queries = np.arange(0, len(sensor_df), 250)
    train_models([TARGET_SENSOR], 2, query_indices=selected_queries, confidence=CONFIDENCE, grid_size=GRID_SIZE)
else:
    print("Batch export skipped. Set RUN_BATCH_EXPORT = True to generate selected-query model files.")